In [1]:
import pandas as pd
import numpy as np

In [59]:
# Load Users dataset
users_df = pd.read_csv("users.csv")
users_df

,user_id,signup_date,country,subscription_plan,subscription_status
0,U00001,2025-09-26,United States,Premium,Cancelled
1,U00002,2025-07-06,United States,Premium,Active
2,U00003,2025-09-27,United Kingdom,Standard,Active
3,U00004,2025-11-07,Canada,Basic,Active
4,U00005,2025-09-02,Canada,Standard,Active
...,...,...,...,...,...
1995,U01996,2025-11-12,Canada,Basic,Cancelled
1996,U01997,2025-06-14,India,Premium,Cancelled
1997,U01998,2025-06-10,United States,Basic,Active
1998,U01999,2025-05-05,United Kingdom,Premium,Active


In [60]:
# Load Content dataset
content_df = pd.read_csv("content_cleaned.csv")
content_df

,show_id,type,title,director,country,release_year,rating,listed_in,duration_value,duration_unit,content_id
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,United States,2020,PG-13,Documentaries,90.0,Minutes,C00001
1,s2,TV Show,Blood & Water,NaN,South Africa,2021,TV-MA,"International TV Shows, TV Dramas, TV Mysteries",2.0,Seasons,C00002
2,s3,TV Show,Ganglands,Julien Leclercq,Unknown,2021,TV-MA,"Crime TV Shows, International TV Shows, TV Act...",1.0,Seasons,C00003
3,s4,TV Show,Jailbirds New Orleans,NaN,Unknown,2021,TV-MA,"Docuseries, Reality TV",1.0,Seasons,C00004
4,s5,TV Show,Kota Factory,NaN,India,2021,TV-MA,"International TV Shows, Romantic TV Shows, TV ...",2.0,Seasons,C00005
...,...,...,...,...,...,...,...,...,...,...,...
8802,s8803,Movie,Zodiac,David Fincher,United States,2007,R,"Cult Movies, Dramas, Thrillers",158.0,Minutes,C08803
8803,s8804,TV Show,Zombie Dumb,NaN,Unknown,2018,TV-Y7,"Kids' TV, Korean TV Shows, TV Comedies",2.0,Seasons,C08804
8804,s8805,Movie,Zombieland,Ruben Fleischer,United States,2009,R,"Comedies, Horror Movies",88.0,Minutes,C08805
8805,s8806,Movie,Zoom,Peter Hewitt,United States,2006,PG,"Children & Family Movies, Comedies",88.0,Minutes,C08806


In [61]:
print("Users:", users_df.shape)
print("Content:", content_df.shape)

Users: (2000, 5)
Content: (8807, 11)


In [62]:
num_watch_records = 50000

print("Number of watch records:", num_watch_records)

Number of watch records: 50000


In [63]:
# Generate user IDs for watch history
watch_user_ids = np.random.choice(
    users_df["user_id"],
    size=num_watch_records
)


In [64]:
print("\nSample Watch User IDs:")
print(watch_user_ids[:10])


Sample Watch User IDs:
['U01210' 'U00518' 'U00693' 'U00356' 'U01594' 'U01791' 'U01981' 'U00227'
 'U01975' 'U00998']


In [65]:
# Generate content IDs for watch history
watch_content_ids = np.random.choice(
    content_df["content_id"],
    size=num_watch_records
)
print("\nSample Watch Content IDs:")
print(watch_content_ids[:10])


Sample Watch Content IDs:
['C00226' 'C07382' 'C07813' 'C03280' 'C02481' 'C08343' 'C04613' 'C05051'
 'C00637' 'C08341']


In [66]:
# Create a lookup for each user's signup date

In [67]:
signup_lookup = users_df.set_index("user_id")["signup_date"]


In [68]:
# Get signup date for each watch record
watch_signup_dates = pd.to_datetime(
    signup_lookup.loc[watch_user_ids].values
)

In [101]:

# Generate dates across the full year
start_date = pd.Timestamp("2025-01-01")
end_date = pd.Timestamp("2025-12-31")

signup_dates = pd.to_datetime(watch_signup_dates)

# Choose a random date across the year for every watch
random_days = np.random.randint(
    0, (end_date - start_date).days + 1,
    size=len(signup_dates)
)

watch_dates = start_date + pd.to_timedelta(random_days, unit="D")

# Enforce the signup-date rule
watch_dates = pd.to_datetime(watch_dates)
watch_dates = pd.DatetimeIndex([
    max(watch_date, signup_date)
    for watch_date, signup_date in zip(watch_dates, signup_dates)
])

print("Earliest:", watch_dates.min())
print("Latest:", watch_dates.max())
print("\nMonthly distribution:")
print(watch_dates.month.value_counts().sort_index())
print("\nWatches before signup:",
      sum(w < s for w, s in zip(watch_dates, signup_dates)))




Earliest: 2025-01-03 00:00:00
Latest: 2025-12-31 00:00:00

Monthly distribution:
1      370
2      988
3     1804
4     2435
5     3355
6     3764
7     4515
8     5325
9     6236
10    6440
11    7187
12    7581
Name: count, dtype: int64

Watches before signup: 0


In [70]:

for name in [
    "watch_ids",
    "selected_user_ids",
    "selected_content_ids",
    "watch_dates",
    "watch_durations",
    "completion_percentages",
    "device_types"
]:
    print(name, "exists:", name in globals())


watch_ids exists: True
selected_user_ids exists: True
selected_content_ids exists: True
watch_dates exists: True
watch_durations exists: True
completion_percentages exists: True
device_types exists: True


In [71]:
# Create watch dates
watch_dates = watch_signup_dates + pd.to_timedelta(
    days_after_signup,
    unit="D"
)

In [72]:
# Keep dates within 2025
watch_dates = pd.Series(watch_dates)
watch_dates = watch_dates.clip(
    upper=pd.Timestamp("2025-12-31")
)

In [73]:
print("\nSample Watch Dates:")
print(watch_dates.head(10))


Sample Watch Dates:
0   2025-12-10
1   2025-11-14
2   2025-10-04
3   2025-11-14
4   2025-09-26
5   2025-12-06
6   2025-11-09
7   2025-09-08
8   2025-11-22
9   2025-07-04
dtype: datetime64[ns]


In [74]:
# Generate watch duration in minutes
watch_duration_minutes = np.random.gamma(
    shape=2.5,
    scale=25,
    size=num_watch_records
)

In [75]:
# Keep values between 5 and 180 minutes
watch_duration_minutes = np.clip(
    watch_duration_minutes,
    5,
    180
)


In [76]:
# Convert to whole numbers
watch_duration_minutes = watch_duration_minutes.astype(int)

In [77]:
print("\nSample Watch Durations:")
print(watch_duration_minutes[:10])


Sample Watch Durations:
[52 16 30 84 69 18 17  5 82 92]


In [78]:
print("\nDuration Statistics:")
print(pd.Series(watch_duration_minutes).describe())


Duration Statistics:
count    50000.000000
mean        61.714660
std         38.058059
min          5.000000
25%         33.000000
50%         54.000000
75%         83.000000
max        180.000000
dtype: float64


In [79]:
# Generate completion percentage
completion_percentage = np.random.beta(
    a=4,
    b=2,
    size=num_watch_records
) * 100


In [80]:
# Keep values between 5% and 100%
completion_percentage = np.clip(
    completion_percentage,
    5,
    100
)

In [81]:
# Round to 2 decimal places
completion_percentage = np.round(
    completion_percentage,
    2
)

In [82]:
print("\nSample Completion Percentages:")
print(completion_percentage[:10])


Sample Completion Percentages:
[32.56 85.69 60.84 77.33 84.11 48.99 68.41 65.35 73.95 71.39]


In [83]:
print("\nCompletion Statistics:")
print(pd.Series(completion_percentage).describe())


Completion Statistics:
count    50000.000000
mean        66.586034
std         17.849585
min          5.360000
25%         54.390000
50%         68.520000
75%         80.582500
max         99.920000
dtype: float64


In [84]:
# Generate device types
devices = [
    "Mobile",
    "Web",
    "TV",
    "Tablet"
]

device_probabilities = [
    0.30,
    0.25,
    0.35,
    0.10
]

device_types = np.random.choice(
    devices,
    size=num_watch_records,
    p=device_probabilities
)

In [85]:
print("\nDevice Distribution:")
print(pd.Series(device_types).value_counts())


Device Distribution:
TV        17453
Mobile    15050
Web       12536
Tablet     4961
Name: count, dtype: int64


In [86]:
# Create Watch History DataFrame
watch_history_df = pd.DataFrame({
    "user_id": watch_user_ids,
    "content_id": watch_content_ids,
    "watch_date": watch_dates,
    "watch_duration_minutes": watch_duration_minutes,
    "completion_percentage": completion_percentage,
    "device_type": device_types
})

In [87]:
# Create unique watch IDs
watch_history_df.insert(
    0,
    "watch_id",
    [f"W{i:06d}" for i in range(1, num_watch_records + 1)]
)

In [88]:
# Display first 10 records
print("\nWatch History:")
print(watch_history_df.head(10))


Watch History:
  watch_id user_id content_id watch_date  watch_duration_minutes  \
0  W000001  U01210     C00226 2025-12-10                      52   
1  W000002  U00518     C07382 2025-11-14                      16   
2  W000003  U00693     C07813 2025-10-04                      30   
3  W000004  U00356     C03280 2025-11-14                      84   
4  W000005  U01594     C02481 2025-09-26                      69   
5  W000006  U01791     C08343 2025-12-06                      18   
6  W000007  U01981     C04613 2025-11-09                      17   
7  W000008  U00227     C05051 2025-09-08                       5   
8  W000009  U01975     C00637 2025-11-22                      82   
9  W000010  U00998     C08341 2025-07-04                      92   

   completion_percentage device_type  
0                  32.56          TV  
1                  85.69      Mobile  
2                  60.84          TV  
3                  77.33      Mobile  
4                  84.11          TV  
5

In [89]:
print("\nShape:")
print(watch_history_df.shape)


Shape:
(50000, 7)


In [90]:
print("\nDuplicate Watch IDs:")
print(watch_history_df["watch_id"].duplicated().sum())


Duplicate Watch IDs:
0


In [91]:
print("\nMissing Values:")
print(watch_history_df.isnull().sum())


Missing Values:
watch_id                  0
user_id                   0
content_id                0
watch_date                0
watch_duration_minutes    0
completion_percentage     0
device_type               0
dtype: int64


In [92]:
# Check whether all user IDs exist in Users
invalid_users = ~watch_history_df["user_id"].isin(users_df["user_id"])

In [93]:
print("\nInvalid User IDs:")
print(invalid_users.sum())



Invalid User IDs:
0


In [94]:
# Check whether all content IDs exist in Content
invalid_content = ~watch_history_df["content_id"].isin(content_df["content_id"])

In [95]:
print("\nInvalid Content IDs:")
print(invalid_content.sum())



Invalid Content IDs:
0


In [96]:
# Check completion percentage
print("\nInvalid Completion Percentages:")
print(
    ((watch_history_df["completion_percentage"] < 0) |
     (watch_history_df["completion_percentage"] > 100)).sum()
)


Invalid Completion Percentages:
0


In [43]:
# Check watch duration
print("\nInvalid Watch Durations:")
print(
    (watch_history_df["watch_duration_minutes"] <= 0).sum()
)


Invalid Watch Durations:
0


In [44]:
watch_history_df.to_csv(
    "watch_history.csv",
    index=False
)

print("\nWatch History dataset saved successfully!")


Watch History dataset saved successfully!


In [45]:
print(watch_history_df.shape)
print(watch_history_df.columns.tolist())
print(watch_history_df.head())

(50000, 7)
['watch_id', 'user_id', 'content_id', 'watch_date', 'watch_duration_minutes', 'completion_percentage', 'device_type']
  watch_id user_id content_id watch_date  watch_duration_minutes  \
0  W000001  U01350     C00180 2025-12-27                      63   
1  W000002  U00551     C03046 2025-06-07                     119   
2  W000003  U01810     C06142 2025-11-28                      76   
3  W000004  U01240     C00228 2025-10-08                     115   
4  W000005  U00012     C04311 2025-11-15                      95   

   completion_percentage device_type  
0                  42.36          TV  
1                  42.83          TV  
2                  63.11      Mobile  
3                  78.93          TV  
4                  93.78          TV  


In [48]:
import os
import pandas as pd

project_path = r"C:\Users\ACER\OneDrive\PROJECTS\Khushi_Work\SQL_PROJECTS\Streaming_Platform_User_Engagement_Retention"
sql_file = os.path.join(project_path, "04_SQL", "05_insert_watch_history.sql")

# Use the existing Watch History DataFrame
watch_df = watch_history_df.copy()

# Convert watch duration to whole minutes for SQL INT
watch_df["watch_duration_minutes"] = (
    watch_df["watch_duration_minutes"].round().astype(int)
)

# Ensure dates are in SQL-compatible format
watch_df["watch_date"] = pd.to_datetime(
    watch_df["watch_date"]
).dt.strftime("%Y-%m-%d")

sql_columns = [
    "watch_id",
    "user_id",
    "content_id",
    "watch_date",
    "watch_duration_minutes",
    "completion_percentage",
    "device_type"
]

def sql_value(value):
    if pd.isna(value):
        return "NULL"
    if isinstance(value, str):
        return "'" + value.replace("'", "''") + "'"
    return str(value)

with open(sql_file, "w", encoding="utf-8") as f:
    f.write("USE StreamingPlatformDB;\nGO\n\n")

    for start in range(0, len(watch_df), 500):
        batch = watch_df.iloc[start:start + 500]

        f.write(
            "INSERT INTO dbo.WatchHistory (" +
            ", ".join(sql_columns) +
            ")\nVALUES\n"
        )

        rows = []
        for _, row in batch[sql_columns].iterrows():
            values = [sql_value(row[col]) for col in sql_columns]
            rows.append("(" + ", ".join(values) + ")")

        f.write(",\n".join(rows) + ";\nGO\n\n")

print("WatchHistory SQL file created successfully!")
print("File:", sql_file)
print("Records:", len(watch_df))

WatchHistory SQL file created successfully!
File: C:\Users\ACER\OneDrive\PROJECTS\Khushi_Work\SQL_PROJECTS\Streaming_Platform_User_Engagement_Retention\04_SQL\05_insert_watch_history.sql
Records: 50000


In [49]:
import os

print("Current folder:", os.getcwd())
print("\nFiles and folders here:")
print(os.listdir())

Current folder: C:\Users\ACER

Files and folders here:
['.anaconda', '.android', '.cache', '.claude', '.claude.json', '.conda', '.condarc', '.continuum', '.copilot', '.dbshell', '.gitconfig', '.ipynb_checkpoints', '.ipython', '.local', '.matplotlib', '.mongorc.js', '.ms-ad', '.streamlit', '.vscode', '.vscode-shared', '01_content_cleaning.ipynb', '02_generate_users.ipynb', '03_generate_watch_history.ipynb', '04_prepare_sql_import.ipynb', 'accessdatabaseengine_X64.exe', 'AppData', 'Application Data', 'Contacts', 'content_cleaned.csv', 'Cookies', 'Documents', 'Favorites', 'IntelGraphicsProfiles', 'Links', 'Local Settings', 'Microsoft', 'My Documents', 'netflix_titles.csv', 'NetHood', 'NTUSER.DAT', 'ntuser.dat.LOG1', 'ntuser.dat.LOG2', 'NTUSER.DAT{e59244eb-2dd9-11f1-a0a9-afa397b7173b}.TM.blf', 'NTUSER.DAT{e59244eb-2dd9-11f1-a0a9-afa397b7173b}.TM.blf.cnpf', 'NTUSER.DAT{e59244eb-2dd9-11f1-a0a9-afa397b7173b}.TMContainer00000000000000000001.regtrans-ms', 'NTUSER.DAT{e59244eb-2dd9-11f1-a0a9-afa

In [50]:


import pandas as pd
import numpy as np

# Load the CSV files from the current folder
users = pd.read_csv("users.csv")
content = pd.read_csv("content_cleaned.csv")

# Check the datasets
print("Users shape:", users.shape)
print("Content shape:", content.shape)

print(users.head())

Users shape: (2000, 5)
Content shape: (8807, 11)
  user_id signup_date         country subscription_plan subscription_status
0  U00001  2025-09-26   United States           Premium           Cancelled
1  U00002  2025-07-06   United States           Premium              Active
2  U00003  2025-09-27  United Kingdom          Standard              Active
3  U00004  2025-11-07          Canada             Basic              Active
4  U00005  2025-09-02          Canada          Standard              Active


In [51]:
print("Shape:", watch_df.shape)
print("Missing dates:", watch_df["watch_date"].isna().sum())
print(
    "December 31 records:",
    (pd.to_datetime(watch_df["watch_date"]) == pd.Timestamp("2025-12-31")).sum()
)

Shape: (50000, 7)
Missing dates: 0
December 31 records: 677


In [52]:
required_variables = [
    "num_watch_records",
    "watch_ids",
    "selected_user_ids",
    "selected_content_ids",
    "watch_signup_dates",
    "watch_dates",
    "watch_durations",
    "completion_percentages",
    "device_types"
]

for name in required_variables:
    print(name, ":", "Available" if name in globals() else "Missing")

num_watch_records : Available
watch_ids : Available
selected_user_ids : Missing
selected_content_ids : Missing
watch_signup_dates : Available
watch_dates : Available
watch_durations : Missing
completion_percentages : Missing
device_types : Available


In [53]:
# Recreate selected users and content for 50,000 watch records

selected_user_ids = np.random.choice(
    users["user_id"],
    size=num_watch_records
)

selected_content_ids = np.random.choice(
    content["content_id"],
    size=num_watch_records
)

# Generate watch durations between 5 and 180 minutes
watch_durations = np.random.gamma(
    shape=2.5,
    scale=25,
    size=num_watch_records
)

watch_durations = np.clip(
    watch_durations, 5, 180
).astype(int)

# Generate completion percentages
completion_percentages = np.random.beta(
    a=4,
    b=2,
    size=num_watch_records
) * 100

completion_percentages = np.clip(
    completion_percentages, 5, 100
).round(2)

print("User IDs:", len(selected_user_ids))
print("Content IDs:", len(selected_content_ids))
print("Watch durations:", len(watch_durations))
print("Completion percentages:", len(completion_percentages))

User IDs: 50000
Content IDs: 50000
Watch durations: 50000
Completion percentages: 50000


In [54]:
# Rebuild the watch-history DataFrame with corrected dates

watch_df = pd.DataFrame({
    "watch_id": watch_ids,
    "user_id": selected_user_ids,
    "content_id": selected_content_ids,
    "watch_date": watch_dates,
    "watch_duration_minutes": watch_durations,
    "completion_percentage": completion_percentages,
    "device_type": device_types
})

print("Shape:", watch_df.shape)
print("Missing values:\n", watch_df.isna().sum())
print(
    "December 31 records:",
    (pd.to_datetime(watch_df["watch_date"]) == pd.Timestamp("2025-12-31")).sum()
)
print(watch_df.head())


Shape: (50000, 7)
Missing values:
 watch_id                  0
user_id                   0
content_id                0
watch_date                0
watch_duration_minutes    0
completion_percentage     0
device_type               0
dtype: int64
December 31 records: 677
  watch_id user_id content_id watch_date  watch_duration_minutes  \
0  W000001  U01340     C00331 2025-12-27                      49   
1  W000002  U00498     C06258 2025-06-07                      87   
2  W000003  U00408     C04144 2025-11-28                      20   
3  W000004  U01516     C02292 2025-10-08                      81   
4  W000005  U01699     C08353 2025-11-15                      44   

   completion_percentage device_type  
0                  74.51          TV  
1                  71.68          TV  
2                  84.14      Mobile  
3                  80.19          TV  
4                  48.32          TV  


In [55]:
watch_df["watch_date"] = pd.to_datetime(
    watch_df["watch_date"]
).dt.strftime("%Y-%m-%d")

watch_df.to_csv("watch_history.csv", index=False)

print("Corrected watch_history.csv saved successfully!")

Corrected watch_history.csv saved successfully!


In [56]:
print(
    (pd.to_datetime(watch_df["watch_date"]) ==
     pd.Timestamp("2025-12-31")).sum()
)

677


In [57]:
saved_df = pd.read_csv("watch_history.csv")

print("Shape:", saved_df.shape)
print(
    "December 31 records:",
    (pd.to_datetime(saved_df["watch_date"]) ==
     pd.Timestamp("2025-12-31")).sum()
)

Shape: (50000, 7)
December 31 records: 677


In [102]:

watch_df = pd.DataFrame({
    "watch_id": watch_ids,
    "user_id": selected_user_ids,
    "content_id": selected_content_ids,
    "watch_date": watch_dates,
    "watch_duration_minutes": watch_durations,
    "completion_percentage": completion_percentages,
    "device_type": device_types
})

print("Shape:", watch_df.shape)
print("Missing values:", watch_df.isnull().sum().sum())
print("\nDecember 31 records:",
      (pd.to_datetime(watch_df["watch_date"]) ==
       pd.Timestamp("2025-12-31")).sum())



Shape: (50000, 7)
Missing values: 0

December 31 records: 133


In [103]:

import os

project_path = r"C:\Users\ACER\OneDrive\PROJECTS\Khushi_Work\SQL_PROJECTS\Streaming_Platform_User_Engagement_Retention"
csv_path = os.path.join(
    project_path, "02_Data", "Cleaned", "watch_history.csv"
)

watch_df["watch_date"] = pd.to_datetime(
    watch_df["watch_date"]
).dt.strftime("%Y-%m-%d")

watch_df.to_csv(csv_path, index=False, quoting=1)

print("Corrected CSV saved successfully!")
print("Location:", csv_path)
print("Shape:", watch_df.shape)


Corrected CSV saved successfully!
Location: C:\Users\ACER\OneDrive\PROJECTS\Khushi_Work\SQL_PROJECTS\Streaming_Platform_User_Engagement_Retention\02_Data\Cleaned\watch_history.csv
Shape: (50000, 7)


In [104]:

import os
import pandas as pd

project_path = r"C:\Users\ACER\OneDrive\PROJECTS\Khushi_Work\SQL_PROJECTS\Streaming_Platform_User_Engagement_Retention"
sql_file = os.path.join(
    project_path, "04_SQL", "05_insert_watch_history.sql"
)

columns = [
    "watch_id", "user_id", "content_id", "watch_date",
    "watch_duration_minutes", "completion_percentage", "device_type"
]

def sql_value(value):
    if pd.isna(value):
        return "NULL"
    if isinstance(value, str):
        return "'" + value.replace("'", "''") + "'"
    return str(value)

with open(sql_file, "w", encoding="utf-8") as f:
    f.write("USE StreamingPlatformDB;\nGO\n\n")

    for start in range(0, len(watch_df), 500):
        batch = watch_df.iloc[start:start + 500]
        f.write(
            "INSERT INTO dbo.WatchHistory (" +
            ", ".join(columns) + ")\nVALUES\n"
        )

        rows = []
        for _, row in batch[columns].iterrows():
            rows.append(
                "(" + ", ".join(sql_value(v) for v in row) + ")"
            )

        f.write(",\n".join(rows) + ";\nGO\n\n")

print("SQL file regenerated successfully!")
print("Records:", len(watch_df))
print("File:", sql_file)


SQL file regenerated successfully!
Records: 50000
File: C:\Users\ACER\OneDrive\PROJECTS\Khushi_Work\SQL_PROJECTS\Streaming_Platform_User_Engagement_Retention\04_SQL\05_insert_watch_history.sql
